# Brain MRI FastAPI Backend — Project-Compatible Colab Notebook

This notebook creates and runs a FastAPI backend using the **actual project model and layout**:
- Google Drive root: `BrainMRI_Data`
- Registered segmentation model: `seg-model v1`
- Export created by `09_m4_t01_export_model.ipynb`: `models/seg-model-v1-torchscript`
- Modalities: `t1`, `t1ce`, `t2`, `flair` (cleaned/standardized NIfTI files)
- Regions and labels: `TC = {1,3}`, `WT = {1,2,3}`, `ET = {3}`

## API requirements covered
- `GET /health`
- `POST /predict`: one scan with four modality uploads; refuses unsupported file types and validates 3D geometry.
- `POST /predict/longitudinal`: two or more visits for the same patient; predicts each visit and calculates percentage volume changes between consecutive visits.
- JSON response includes segmentation mask path, tumor volumes, timings, radiomics status, and longitudinal changes when applicable.
- Includes a small API test for wrong file types.

**Important:** Upload cleaned/standardized scans matching the project's model input conventions. This is not a raw DICOM conversion or a clinical diagnostic tool. The notebook checks for the registered TorchScript export; if it is missing, it runs the repository's `scripts/export_model.py` using the verified model checkpoint. It does not retrain the model.


In [13]:
# 1. Mount Drive and configure the project paths
from pathlib import Path
import os, sys, json, subprocess, textwrap, time

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
REPO = Path("/content/repo")
MODEL_DIR = BASE / "runs/M2-T04_nnunet/nnUNet_results/Dataset501_BraTS2021/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres"
EXPORT_DIR = BASE / "models/seg-model-v1-torchscript"
API_DIR = BASE / "api_backend"
OUT_DIR = BASE / "runs/API_Backend"
OUT_DIR.mkdir(parents=True, exist_ok=True)
API_DIR.mkdir(parents=True, exist_ok=True)

print("Drive project exists:", BASE.exists())
print("Model directory:", MODEL_DIR)
print("Export directory:", EXPORT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project exists: True
Model directory: /content/drive/MyDrive/BrainMRI_Data/runs/M2-T04_nnunet/nnUNet_results/Dataset501_BraTS2021/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres
Export directory: /content/drive/MyDrive/BrainMRI_Data/models/seg-model-v1-torchscript


In [14]:
# 2. Get the same repository branch used by the project's M4 export notebook
BRANCH = "main"
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/nhahub/NHA-5-187.git", str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))
print("Repository ready:", REPO)
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip())


Repository ready: /content/repo
71e78f2


In [15]:
# 3. Install core inference/API dependencies; PyRadiomics is handled separately as optional.
%pip -q install "nnunetv2==2.8.1" "fastapi>=0.115,<1.0" "uvicorn[standard]>=0.30,<1.0" python-multipart nibabel SimpleITK pandas requests


In [16]:
# 3a. Optional PyRadiomics install.
# PyPI source builds can fail on Colab's Python 3.13. Try the project's previously used GitHub source.
import subprocess, sys
rad_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/AIM-Harvard/pyradiomics.git"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(rad_install.stdout[-5000:] if rad_install.stdout else "(no installer output)")
try:
    import radiomics
    RADIOMICS_AVAILABLE = True
    print("PyRadiomics available:", radiomics.__version__)
except Exception as exc:
    RADIOMICS_AVAILABLE = False
    print("PyRadiomics unavailable; segmentation API will still run without radiomics.")
    print(type(exc).__name__, str(exc))


(no installer output)
PyRadiomics available: 3.1.1.dev111+g8ed579383


In [17]:
# 4. Configure nnU-Net and verify the registered seg-model v1 checkpoint against the repository manifest
import hashlib

for key in ("nnUNet_raw", "nnUNet_preprocessed"):
    p = Path("/content/nnunet_work") / key
    p.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(p)
os.environ["nnUNet_results"] = str(BASE / "runs/M2-T04_nnunet/nnUNet_results")

manifest_path = REPO / "models/registry/seg-model/v1/manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Registered model manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
checkpoint_rel = manifest.get("nnunet", {}).get("checkpoint", "checkpoint_best.pth")
checkpoint = MODEL_DIR / "fold_0" / checkpoint_rel
if not checkpoint.is_file():
    raise FileNotFoundError(f"Registered checkpoint missing: {checkpoint}. Complete M2-T04 setup first.")

h = hashlib.sha256()
with checkpoint.open("rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        h.update(block)
checkpoint_sha = h.hexdigest()
expected_sha = manifest.get("checkpoint_file", {}).get("sha256")
if expected_sha and checkpoint_sha != expected_sha:
    raise RuntimeError("Checkpoint SHA-256 mismatch: refusing to export an unverified model.")
print("Registered seg-model v1 checkpoint verified:", checkpoint.name)
print("Checkpoint SHA-256:", checkpoint_sha)


Registered seg-model v1 checkpoint verified: checkpoint_best.pth
Checkpoint SHA-256: 7dcb450bf5455a0c62c9e1b368ee6341c687f555528d5ed3cd3399d88ea0f5e0


In [18]:
# 5. Verify/reuse the TorchScript export or run export with full diagnostics
from bmts.deploy.export import sha256

export_manifest_path = EXPORT_DIR / "deploy_manifest.json"
export_ok = False
if export_manifest_path.is_file():
    try:
        em = json.loads(export_manifest_path.read_text(encoding="utf-8"))
        model_file = EXPORT_DIR / em["model_file"]
        export_ok = (
            em.get("source", {}).get("checkpoint_sha256") == checkpoint_sha
            and model_file.is_file()
            and sha256(model_file) == em.get("model_sha256")
        )
    except Exception as exc:
        print("Existing export manifest is invalid:", repr(exc))
        export_ok = False

if not export_ok:
    EXPORT_DIR.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, str(REPO / "scripts/export_model.py"),
        "--model-dir", str(MODEL_DIR), "--out", str(EXPORT_DIR)
    ]
    print("Exporting registered model for API inference...")
    export_result = subprocess.run(
        cmd, cwd=str(REPO), text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(export_result.stdout[-12000:] if export_result.stdout else "(export script produced no output)")
    if export_result.returncode != 0:
        raise RuntimeError(
            "TorchScript export failed (exit code {}). The complete exporter log is printed above. "
            "Do not continue to the API cells until this is resolved.".format(export_result.returncode)
        )
else:
    print("Verified TorchScript export already exists; reusing it.")

if not export_manifest_path.is_file():
    raise FileNotFoundError(
        f"Export script finished without creating the expected manifest: {export_manifest_path}"
    )
em = json.loads(export_manifest_path.read_text(encoding="utf-8"))
print("Export ready:", EXPORT_DIR / em["model_file"])


Verified TorchScript export already exists; reusing it.
Export ready: /content/drive/MyDrive/BrainMRI_Data/models/seg-model-v1-torchscript/seg_model.torchscript.pt


## 6. Write the FastAPI application

The backend uses `FastSegmenter` from the team's `bmts.deploy.segmenter` module, matching the M4 export notebook. It accepts only `.nii` / `.nii.gz` uploads, validates 3D images and matching geometry, and stores each request in its own folder to avoid overwriting concurrent uploads. Radiomics is optional: if extraction fails, the JSON reports that status without hiding a successful segmentation.


In [19]:
APP_SOURCE = r'''from __future__ import annotations

import json
import os
import re
import shutil
import uuid
from pathlib import Path
from typing import Any

import nibabel as nib
import numpy as np
import pandas as pd
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import JSONResponse

BASE = Path(os.getenv("BRAINMRI_BASE", "/content/drive/MyDrive/BrainMRI_Data"))
EXPORT_DIR = Path(os.getenv("BRAINMRI_EXPORT_DIR", str(BASE / "models/seg-model-v1-torchscript")))
OUTPUT_ROOT = Path(os.getenv("BRAINMRI_API_OUTPUT", str(BASE / "runs/API_Backend")))
DEVICE = os.getenv("BRAINMRI_DEVICE", "cuda" if __import__("torch").cuda.is_available() else "cpu")
RUN_RADIOMICS_DEFAULT = os.getenv("API_RUN_RADIOMICS", "true").lower() in {"1", "true", "yes"}
# TTA can multiply inference time substantially. It is disabled by default; set API_TTA=true to enable it.
TTA = os.getenv("API_TTA", "false").lower() in {"1", "true", "yes"}
MAX_UPLOAD_MB = int(os.getenv("API_MAX_UPLOAD_MB", "512"))
MODALITIES = ("t1", "t1ce", "t2", "flair")
REGIONS = {"TC": (1, 3), "WT": (1, 2, 3), "ET": (3,)}

app = FastAPI(
    title="Brain MRI Segmentation API",
    version="1.0.1",
    description="Project-compatible API for registered seg-model v1. Research prototype only."
)

_segmenter = None

def safe_id(value: str | None) -> str:
    value = (value or "case").strip()
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("._-")
    return (value or "case")[:80]

def is_nii_name(filename: str | None) -> bool:
    name = (filename or "").lower()
    return name.endswith(".nii") or name.endswith(".nii.gz")

def get_segmenter():
    global _segmenter
    if _segmenter is None:
        if not EXPORT_DIR.is_dir() or not (EXPORT_DIR / "deploy_manifest.json").is_file():
            raise RuntimeError(f"Registered TorchScript export is missing: {EXPORT_DIR}")
        from bmts.deploy.segmenter import FastSegmenter
        _segmenter = FastSegmenter.from_dir(EXPORT_DIR, device=DEVICE, tta=TTA)
    return _segmenter

async def save_upload(upload: UploadFile, dest: Path) -> None:
    if not is_nii_name(upload.filename):
        raise HTTPException(status_code=415, detail=f"Unsupported file type for {upload.filename!r}; upload .nii or .nii.gz.")
    total = 0
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        with dest.open("wb") as out:
            while True:
                chunk = await upload.read(1024 * 1024)
                if not chunk:
                    break
                total += len(chunk)
                if total > MAX_UPLOAD_MB * 1024 * 1024:
                    raise HTTPException(status_code=413, detail=f"File exceeds {MAX_UPLOAD_MB} MB limit.")
                out.write(chunk)
    finally:
        await upload.close()
    if total == 0:
        dest.unlink(missing_ok=True)
        raise HTTPException(status_code=400, detail=f"Empty upload: {upload.filename!r}")

def validate_modalities(paths: dict[str, Path]) -> dict[str, Any]:
    images = {}
    try:
        for name in MODALITIES:
            p = paths[name]
            try:
                img = nib.load(str(p))
                if len(img.shape) != 3:
                    raise ValueError(f"{name} must be a 3D NIfTI image; got shape {img.shape}.")
                if not np.all(np.isfinite(img.affine)):
                    raise ValueError(f"{name} has a non-finite affine matrix.")
                images[name] = img
            except Exception as exc:
                raise ValueError(f"Invalid or unreadable NIfTI for {name}: {exc}") from exc
        ref = images["t1"]
        for name, img in images.items():
            if img.shape[:3] != ref.shape[:3] or not np.allclose(img.affine, ref.affine, atol=1e-4):
                raise ValueError(f"Geometry mismatch: {name} shape/affine differs from t1.")
        voxel_volume_mm3 = float(abs(np.linalg.det(ref.affine[:3, :3])))
        return {
            "shape": [int(x) for x in ref.shape],
            "voxel_volume_mm3": voxel_volume_mm3,
            "voxel_spacing_mm": [float(x) for x in ref.header.get_zooms()[:3]],
            "geometry_validated": True,
        }
    finally:
        images.clear()

def normalize_segmentation_axes(seg: np.ndarray, reference_shape: tuple[int, ...]) -> np.ndarray:
    """Return the prediction in the reference NIfTI array order (i, j, k).

    Some inference wrappers return arrays in z,y,x order. Only reverse all axes
    when the shape exactly matches the reversed reference shape; never guess for
    other shapes.
    """
    arr = np.asarray(seg)
    target = tuple(int(x) for x in reference_shape[:3])
    if arr.shape == target:
        return arr
    if arr.ndim == 3 and arr.shape == target[::-1]:
        return np.transpose(arr, (2, 1, 0))
    raise ValueError(f"Prediction shape {arr.shape} does not match input NIfTI shape {target} or its reversed order.")

def calculate_volumes(mask_path: Path, voxel_volume_mm3: float) -> dict[str, Any]:
    mask_img = nib.load(str(mask_path))
    mask = np.asarray(mask_img.dataobj)
    if mask.ndim != 3:
        raise ValueError("Predicted mask is not 3D.")
    result = {}
    for region, labels in REGIONS.items():
        voxels = int(np.isin(mask, labels).sum())
        result[region] = {
            "voxels": voxels,
            "volume_cm3": round(voxels * voxel_volume_mm3 / 1000.0, 6),
        }
    result["label_counts"] = {str(int(label)): int((mask == label).sum()) for label in np.unique(mask)}
    return result

def extract_radiomics(paths: dict[str, Path], mask_path: Path, out_csv: Path) -> dict[str, Any]:
    try:
        from radiomics import featureextractor
        import SimpleITK as sitk

        mask = sitk.ReadImage(str(mask_path))
        settings = {
            "binWidth": 25,
            "resampledPixelSpacing": None,
            "interpolator": "sitkBSpline",
            "normalize": False,
            "label": 1,
        }
        extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
        rows = []
        for modality in ("t1ce", "t2", "flair"):
            image = sitk.ReadImage(str(paths[modality]))
            # Explicitly check dimensions and physical geometry; do not silently resize/transpose.
            if image.GetSize() != mask.GetSize():
                raise ValueError(
                    f"Radiomics geometry mismatch for {modality}: image size {image.GetSize()} "
                    f"vs mask size {mask.GetSize()}. Both files must be in the same NIfTI geometry "
                    "and must be read from the current request folder."
                )
            if (image.GetOrigin() != mask.GetOrigin() or image.GetSpacing() != mask.GetSpacing()
                    or image.GetDirection() != mask.GetDirection()):
                raise ValueError(f"Radiomics physical geometry mismatch for {modality} image and mask.")
            for label in (1, 2, 3):
                binary = sitk.Cast(mask == label, sitk.sitkUInt8)
                if int(sitk.GetArrayViewFromImage(binary).sum()) == 0:
                    continue
                features = extractor.execute(image, binary, label=1)
                row = {"modality": modality, "label": label}
                row.update({str(k): (float(v) if isinstance(v, (int, float, np.number)) else str(v))
                            for k, v in features.items() if not str(k).startswith("diagnostics_")})
                rows.append(row)
        if not rows:
            return {"status": "empty_mask", "csv_path": None, "rows": 0}
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(out_csv, index=False)
        return {"status": "success", "csv_path": str(out_csv), "rows": len(rows)}
    except Exception as exc:
        return {"status": "failed", "error": f"{type(exc).__name__}: {exc}", "csv_path": None}

def run_one_case(case_id: str, paths: dict[str, Path], case_out: Path, run_radiomics: bool) -> dict[str, Any]:
    validation = validate_modalities(paths)
    case_out.mkdir(parents=True, exist_ok=True)
    mask_path = case_out / f"{case_id}_seg.nii.gz"
    try:
        segmenter = get_segmenter()
        # FastSegmenter expects file paths. Pass plain strings rather than Path objects
        # to avoid path-like tuple/type incompatibilities across repository versions.
        input_files = [str(paths[m]) for m in MODALITIES]
        segment_output = segmenter.segment(input_files)
        if not isinstance(segment_output, tuple) or len(segment_output) < 3:
            raise TypeError(
                "Unexpected FastSegmenter.segment() result. Expected (segmentation, output, timings); "
                f"received {type(segment_output).__name__}."
            )
        seg, _segment_output_path, timings = segment_output
        if isinstance(seg, (str, Path)):
            # Some versions return a saved mask path as the first result.
            seg = np.asarray(nib.load(str(seg)).dataobj)
        elif isinstance(seg, tuple):
            # Do not pass tuple objects to Path/nibabel; only accept a tuple that wraps one array.
            if len(seg) == 1:
                seg = seg[0]
            else:
                raise TypeError(
                    "FastSegmenter returned a tuple for the segmentation array; "
                    f"tuple length={len(seg)}. Check the installed segmenter API."
                )
        ref = nib.load(str(paths["t1"]))
        seg = normalize_segmentation_axes(np.asarray(seg), ref.shape)
        seg_img = nib.Nifti1Image(seg.astype(np.uint8), ref.affine, ref.header.copy())
        nib.save(seg_img, str(mask_path))
    except HTTPException:
        raise
    except Exception as exc:
        raise RuntimeError(f"Segmentation failed: {type(exc).__name__}: {exc}") from exc

    volumes = calculate_volumes(mask_path, validation["voxel_volume_mm3"])
    radiomics = (extract_radiomics(paths, mask_path, case_out / f"{case_id}_radiomics.csv")
                 if run_radiomics else {"status": "disabled", "csv_path": None, "rows": 0})
    return {
        "case_id": case_id,
        "input_validation": validation,
        "segmentation": {"status": "success", "mask_path": str(mask_path), "volumes": volumes},
        "inference_timings": {str(k): float(v) for k, v in timings.items()},
        "radiomics": radiomics,
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "service": "Brain MRI Segmentation API",
        "export_dir_exists": EXPORT_DIR.is_dir(),
        "export_manifest_exists": (EXPORT_DIR / "deploy_manifest.json").is_file(),
        "device": DEVICE,
        "tta_enabled": TTA,
        "supported_file_extensions": [".nii", ".nii.gz"],
        "modalities_required": list(MODALITIES),
    }

@app.post("/predict")
async def predict(
    t1: UploadFile = File(...),
    t1ce: UploadFile = File(...),
    t2: UploadFile = File(...),
    flair: UploadFile = File(...),
    case_id: str = Form("case"),
    run_radiomics: bool = Form(RUN_RADIOMICS_DEFAULT),
):
    cid = safe_id(case_id)
    request_dir = OUTPUT_ROOT / "requests" / f"{cid}_{uuid.uuid4().hex[:8]}"
    inputs_dir = request_dir / "inputs"
    uploads = {"t1": t1, "t1ce": t1ce, "t2": t2, "flair": flair}
    paths = {}
    try:
        for modality, upload in uploads.items():
            suffix = ".nii.gz" if (upload.filename or "").lower().endswith(".nii.gz") else ".nii"
            paths[modality] = inputs_dir / f"{cid}_{modality}{suffix}"
            await save_upload(upload, paths[modality])
        result = run_one_case(cid, paths, request_dir / "results", run_radiomics)
        return JSONResponse({"status": "success", "pipeline": "seg-model-v1-torchscript",
                             "result": result, "longitudinal_changes": None})
    except HTTPException:
        shutil.rmtree(request_dir, ignore_errors=True)
        raise
    except ValueError as exc:
        shutil.rmtree(request_dir, ignore_errors=True)
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    except RuntimeError as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"{type(exc).__name__}: {exc}") from exc

@app.post("/predict/longitudinal")
async def predict_longitudinal(
    patient_id: str = Form(...),
    visits_json: str = Form(..., description='JSON array: [{"visit_id":"preop","t1":"filename.nii.gz","t1ce":"...","t2":"...","flair":"..."}]'),
    files: list[UploadFile] = File(..., description="Upload 4 files per visit, ordered by visits_json: t1, t1ce, t2, flair."),
    run_radiomics: bool = Form(RUN_RADIOMICS_DEFAULT),
):
    """Predict each visit for one patient. Files must be ordered visit-by-visit, 4 modalities per visit."""
    try:
        visits = json.loads(visits_json)
    except Exception as exc:
        raise HTTPException(status_code=422, detail="visits_json must be valid JSON.") from exc
    if not isinstance(visits, list) or len(visits) < 2:
        raise HTTPException(status_code=422, detail="At least two visits are required.")
    if len(files) != 4 * len(visits):
        raise HTTPException(status_code=422, detail=f"Expected {4 * len(visits)} files (4 modalities per visit), received {len(files)}.")
    patient = safe_id(patient_id)
    request_dir = OUTPUT_ROOT / "longitudinal" / f"{patient}_{uuid.uuid4().hex[:8]}"
    outputs = []
    try:
        for i, visit in enumerate(visits):
            if not isinstance(visit, dict):
                raise HTTPException(status_code=422, detail=f"Visit {i} must be a JSON object.")
            visit_id = safe_id(str(visit.get("visit_id", f"visit_{i+1}")))
            visit_dir = request_dir / visit_id
            input_dir = visit_dir / "inputs"
            input_dir.mkdir(parents=True, exist_ok=True)
            paths = {}
            for j, modality in enumerate(MODALITIES):
                upload = files[i * 4 + j]
                if not is_nii_name(upload.filename):
                    raise HTTPException(status_code=415, detail=f"Unsupported file type for {upload.filename!r}; upload .nii or .nii.gz.")
                suffix = ".nii.gz" if (upload.filename or "").lower().endswith(".nii.gz") else ".nii"
                path = input_dir / f"{visit_id}_{modality}{suffix}"
                await save_upload(upload, path)
                paths[modality] = path
            result = run_one_case(f"{patient}_{visit_id}", paths, visit_dir / "results", run_radiomics)
            outputs.append({"visit_id": visit_id, **result})
        changes = []
        for before, after in zip(outputs, outputs[1:]):
            for region in ("TC", "WT", "ET"):
                v0 = before["segmentation"]["volumes"][region]["volume_cm3"]
                v1 = after["segmentation"]["volumes"][region]["volume_cm3"]
                pct = None if v0 == 0 else round((v1 - v0) / v0 * 100.0, 3)
                changes.append({
                    "from_visit": before["visit_id"], "to_visit": after["visit_id"],
                    "region": region, "volume_before_cm3": v0, "volume_after_cm3": v1,
                    "change_cm3": round(v1 - v0, 6), "change_percent": pct,
                    "note": "change_percent is null when baseline volume is zero"
                })
        return JSONResponse({"status": "success", "patient_id": patient,
                             "pipeline": "seg-model-v1-torchscript",
                             "visits": outputs, "longitudinal_changes": changes})
    except HTTPException:
        raise
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"{type(exc).__name__}: {exc}") from exc

@app.get("/")
def root():
    return {"message": "Brain MRI API is running", "docs": "/docs", "health": "/health"}
'''
app_path = API_DIR / 'main.py'
app_path.write_text(APP_SOURCE, encoding='utf-8')
print('Wrote:', app_path)
print('Bytes:', app_path.stat().st_size)


Wrote: /content/drive/MyDrive/BrainMRI_Data/api_backend/main.py
Bytes: 15926


## 7. Add dependencies, environment template, and README

In [20]:
# Create deployment metadata and dependencies.
# PyRadiomics is intentionally optional: Colab/Python 3.13 may fail to build the PyPI source distribution.
requirements = '''fastapi>=0.115,<1.0
uvicorn[standard]>=0.30,<1.0
python-multipart
nnunetv2==2.8.1
nibabel>=5.2
numpy>=1.26
pandas>=2.0
SimpleITK>=2.2.1
'''
(API_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")

env_example = '''BRAINMRI_BASE=/content/drive/MyDrive/BrainMRI_Data
BRAINMRI_EXPORT_DIR=/content/drive/MyDrive/BrainMRI_Data/models/seg-model-v1-torchscript
BRAINMRI_API_OUTPUT=/content/drive/MyDrive/BrainMRI_Data/runs/API_Backend
BRAINMRI_DEVICE=cpu
API_RUN_RADIOMICS=true
API_MAX_UPLOAD_MB=512
'''
(API_DIR / ".env.example").write_text(env_example, encoding="utf-8")

readme = '''# Brain MRI FastAPI Backend (project-compatible)

Uses the registered seg-model v1 TorchScript export and the project label scheme.

## Run in Colab
Run the notebook from top to bottom. It mounts Drive, verifies the registered checkpoint, reuses/creates the TorchScript export, writes the backend, starts Uvicorn, and checks the health endpoint with a startup timeout and captured logs.

## Endpoints
- `GET /health`
- `POST /predict`: file fields `t1`, `t1ce`, `t2`, `flair`, plus `case_id` and `run_radiomics`.
- `POST /predict/longitudinal`: `patient_id`, `visits_json`, and four files per visit ordered by visit and modality.

## Input requirements
Only `.nii` and `.nii.gz` files are accepted. Inputs must be readable 3D NIfTI images with matching dimensions and affine geometry. Upload cleaned/standardized modality files.

## Outputs
Masks, TC/WT/ET volumes, inference timings, optional radiomics CSV, and sequential longitudinal volume changes are saved below `runs/API_Backend/`.

## Limitations
- The model is not retrained.
- Longitudinal percentage change is undefined when the baseline volume is zero.
- Longitudinal scans should be registered/aligned.
- Radiomics is optional and its failure does not suppress a successful segmentation.
- Research prototype only; not for clinical diagnosis.
'''
(API_DIR / "README.md").write_text(readme, encoding="utf-8")
print("Created requirements.txt, .env.example, README.md")


Created requirements.txt, .env.example, README.md


## 8. Validate the application code before starting the server

In [21]:
import ast
ast.parse(app_path.read_text(encoding="utf-8"))
print("Python syntax validation: PASS")
print("Backend files:", sorted(p.name for p in API_DIR.iterdir()))


Python syntax validation: PASS
Backend files: ['.env.example', 'README.md', '__pycache__', 'main.py', 'requirements.txt']


## 9. Start the API server in the background

After this cell finishes, open the interactive docs at the local Colab URL shown by the proxy cell below. The server uses the project Drive export and saves results under `runs/API_Backend/`.


In [22]:
# Start Uvicorn robustly, keep logs, and wait for /health to respond.
import os, sys, time, subprocess, requests
from pathlib import Path

BASE_URL = "http://127.0.0.1:8000"
SERVER_LOG = Path("/content/brainmri_api.log")

# Stop only a server process started by this notebook.
if "api_process" in globals() and api_process is not None and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        api_process.kill()
        api_process.wait(timeout=5)

if "server_log_handle" in globals():
    try:
        server_log_handle.close()
    except Exception:
        pass

server_log_handle = open(SERVER_LOG, "w", buffering=1)
server_env = os.environ.copy()
server_env.update({
    "BRAINMRI_BASE": str(BASE),
    "BRAINMRI_EXPORT_DIR": str(EXPORT_DIR),
    "BRAINMRI_API_OUTPUT": str(OUT_DIR),
    # The exported TorchScript was generated with CPU Torch; use CPU by default for reliable startup.
    "BRAINMRI_DEVICE": "cpu",
    "PYTHONUNBUFFERED": "1",
    "PYTHONPATH": str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
})

api_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app",
     "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"],
    cwd=str(API_DIR),
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    env=server_env,
)

print("Started Uvicorn PID:", api_process.pid)
health_ok = False
last_error = None

for attempt in range(60):
    if api_process.poll() is not None:
        print("❌ Uvicorn exited with code:", api_process.returncode)
        break
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=2)
        if r.status_code == 200:
            print("✅ GET /health:", r.status_code)
            print(r.json())
            health_ok = True
            break
        last_error = f"HTTP {r.status_code}: {r.text[:1000]}"
    except requests.RequestException as exc:
        last_error = f"{type(exc).__name__}: {exc}"
    time.sleep(1)

if not health_ok:
    server_log_handle.flush()
    print("\n❌ API health check did not succeed.")
    print("Last connection result:", last_error)
    print("\n--- Uvicorn log (last 15000 chars) ---")
    print(SERVER_LOG.read_text(errors="replace")[-15000:] if SERVER_LOG.exists() else "(log file missing)")
    raise RuntimeError("FastAPI did not become healthy. Diagnose the Uvicorn log above before continuing.")

print("API server is healthy and listening on port 8000.")


Started Uvicorn PID: 4236
✅ GET /health: 200
{'status': 'ok', 'service': 'Brain MRI Segmentation API', 'export_dir_exists': True, 'export_manifest_exists': True, 'device': 'cpu', 'tta_enabled': False, 'supported_file_extensions': ['.nii', '.nii.gz'], 'modalities_required': ['t1', 't1ce', 't2', 'flair']}
API server is healthy and listening on port 8000.


## 10. Smoke-test health and wrong file type

This test checks the health endpoint and confirms that a `.txt` upload is rejected with HTTP 415. It does not run a full segmentation, because that requires four real cleaned MRI modality files.


In [23]:
# Smoke-test health and unsupported file-type rejection; show server logs if a request fails.
import requests
from pathlib import Path

BASE_URL = "http://127.0.0.1:8000"
try:
    health_response = requests.get(f"{BASE_URL}/health", timeout=10)
    print("GET /health:", health_response.status_code, health_response.text[:2000])
    health_response.raise_for_status()

    bad = {
        "t1": ("t1.txt", b"not a nifti", "text/plain"),
        "t1ce": ("t1ce.nii.gz", b"placeholder", "application/octet-stream"),
        "t2": ("t2.nii.gz", b"placeholder", "application/octet-stream"),
        "flair": ("flair.nii.gz", b"placeholder", "application/octet-stream"),
    }
    bad_response = requests.post(
        f"{BASE_URL}/predict", files=bad,
        data={"case_id": "invalid_type", "run_radiomics": "false"}, timeout=30
    )
    print("Wrong file type status:", bad_response.status_code)
    print("Response:", bad_response.text[:1000])
    assert bad_response.status_code == 415, "API should reject unsupported file types with HTTP 415."
    print("API smoke tests: PASS")
except Exception:
    log_path = Path("/content/brainmri_api.log")
    print("\n--- Uvicorn log (last 15000 chars) ---")
    print(log_path.read_text(errors="replace")[-15000:] if log_path.exists() else "(log file missing)")
    raise


GET /health: 200 {"status":"ok","service":"Brain MRI Segmentation API","export_dir_exists":true,"export_manifest_exists":true,"device":"cpu","tta_enabled":false,"supported_file_extensions":[".nii",".nii.gz"],"modalities_required":["t1","t1ce","t2","flair"]}
Wrong file type status: 415
Response: {"detail":"Unsupported file type for 't1.txt'; upload .nii or .nii.gz."}
API smoke tests: PASS


## 11. Access the interactive docs

In Colab, use the proxy URL for port 8000. The Swagger page lets you upload the four modality files to `/predict`. For a public/shareable deployment, deploy the same `api_backend` folder to a server with the model export and dependencies; a local Colab runtime is temporary.



In [24]:
# Display clickable links directly in the notebook output.
from google.colab.output import eval_js
from IPython.display import display, HTML

BASE_URL = eval_js("google.colab.kernel.proxyPort(8000)")
DOCS_URL = BASE_URL.rstrip("/") + "/docs"
HEALTH_URL = BASE_URL.rstrip("/") + "/health"

print("FastAPI backend is running.")
display(HTML(f'''
<div style="padding:14px;border:1px solid #cbd5e1;border-radius:10px;line-height:2">
  <h3 style="margin:0 0 8px 0">Brain MRI FastAPI Backend</h3>
  <div>🌐 <b>Open API website:</b> <a href="{BASE_URL}" target="_blank">{BASE_URL}</a></div>
  <div>🧪 <b>API documentation (Swagger UI):</b> <a href="{DOCS_URL}" target="_blank">{DOCS_URL}</a></div>
  <div>💚 <b>Health check:</b> <a href="{HEALTH_URL}" target="_blank">{HEALTH_URL}</a></div>
</div>
'''))
print("Note: these links work while this Google Colab runtime and the API server remain active.")


FastAPI backend is running.


Note: these links work while this Google Colab runtime and the API server remain active.


## Expected result

- API backend files are written to `BrainMRI_Data/api_backend/`.
- Run outputs are written to `BrainMRI_Data/runs/API_Backend/`.
- The model checkpoint is verified against `models/registry/seg-model/v1/manifest.json`.
- The exported TorchScript model is reused when valid or created by the repository export script when missing.
- Invalid extensions are rejected with HTTP `415 Unsupported Media Type`.\n- PyRadiomics is optional; if unavailable, segmentation still runs and the response reports radiomics as unavailable.

**Validation boundary:** this notebook performs syntax checks and API smoke tests. A full inference test requires real cleaned NIfTI files from the project and was not run during notebook generation.
